# DevFlow - Lab: Contratos de Dados com Pydantic

**Curso:** Agentic Engineering - SkillGo
**Prof.:** Ives Santos

---

Este notebook isola **uma unica peca** do agente DevFlow: o **contrato de dados**.
Nao ha LLM, grafo, RAG nem chave de API. Roda inteiro no Google Colab em segundos.

> **Ideia central:** num agente, todo dado que entra ou sai passa por um contrato tipado.
> O contrato valida a entrada **antes** de gastar um token, obriga o LLM a responder num
> formato conhecido e entrega a cada etapa um objeto com formato garantido, em vez de texto solto.

Aqui estudamos tres contratos, um de cada ponto do fluxo:

| Contrato | Quem produz | Papel no agente |
|---|---|---|
| `Issue` | usuario / sistema de tickets | entrada do agente |
| `Triagem` | LLM (structured output) | classificacao da issue |
| `DecisaoHumana` | pessoa revisora | resposta na pausa humana, antes de terminar |

```
Issue --> guardrails --> LLM --> Triagem --> ... --> pausa --> DecisaoHumana --> fim
```

## 1. Ambiente

O Colab ja vem com Pydantic, mas este lab depende da **versao 2** (`field_validator`,
`model_dump`, `model_validate_json`). A celula abaixo garante isso.

In [ ]:
!pip install -q "pydantic>=2.7"

import pydantic
print("pydantic", pydantic.VERSION)
assert pydantic.VERSION.startswith("2"), "Este lab exige Pydantic 2"

### 1.1 Imports e um utilitario para ler erros

`ValidationError` e a excecao que o Pydantic levanta quando um dado viola o contrato.
Ela carrega **todos** os erros de uma vez (nao so o primeiro). A funcao `mostrar_erros`
apenas imprime cada um de forma legivel: o campo, o tipo do erro e a mensagem.

In [ ]:
from typing import List, Literal
import json

from pydantic import BaseModel, Field, ValidationError, field_validator


def mostrar_erros(erro: ValidationError) -> None:
    print(f"{erro.error_count()} erro(s) de validacao:")
    for e in erro.errors():
        campo = ".".join(str(p) for p in e["loc"]) or "(modelo)"
        print(f"  - {campo:<22} [{e['type']}] {e['msg']}")

---
## 2. Entrada: `Issue`

A issue ficticia que entra no sistema tem quatro blocos:

- **identidade:** `id`, `projeto`, `autor`, `criada_em`, `labels`
- **titulo**
- **descricao**
- **criterios de aceite**

**Ferramentas usadas:**

| Recurso | O que garante |
|---|---|
| campo sem valor padrao (`projeto: str`) | campo obrigatorio |
| `Field(min_length=5)` | titulo nao pode ser vazio ou "bug" |
| `Field(min_length=20)` | descricao minimamente util |
| `Field(min_length=1)` em lista | pelo menos um criterio de aceite |
| `Field(default_factory=list)` | `labels` e opcional, e cada instancia ganha sua propria lista |
| `@field_validator` | limpa criterios em branco e recusa lista so com espacos |

Os metodos `criterios_numerados()` e `ids_criterios()` criam os identificadores `CA1`, `CA2`...
que o LLM vai **citar** mais adiante no fluxo. Isso permite conferir, por codigo, se todo
criterio foi coberto.

In [ ]:
class Issue(BaseModel):
    """A issue ficticia que entra no sistema."""

    id: str = Field(description="Identificador unico, ex.: ISSUE-1042")
    projeto: str
    autor: str
    criada_em: str
    labels: List[str] = Field(default_factory=list)
    titulo: str = Field(min_length=5)
    descricao: str = Field(min_length=20)
    criterios_aceite: List[str] = Field(min_length=1)

    @field_validator("criterios_aceite")
    @classmethod
    def _criterios_nao_vazios(cls, valor: List[str]) -> List[str]:
        limpos = [c.strip() for c in valor if c and c.strip()]
        if not limpos:
            raise ValueError("a issue precisa de pelo menos um criterio de aceite")
        return limpos

    def criterios_numerados(self) -> List[str]:
        """Devolve os criterios prefixados com CA1, CA2... para o modelo citar."""
        return [f"CA{i}: {c}" for i, c in enumerate(self.criterios_aceite, start=1)]

    def ids_criterios(self) -> List[str]:
        return [f"CA{i}" for i in range(1, len(self.criterios_aceite) + 1)]

### 2.1 Execucao: uma issue valida

Este e o mesmo JSON de `data/issues/ISSUE-1042.json` do projeto. `model_validate` recebe
um `dict` (como viria de uma API ou de um arquivo) e devolve um objeto tipado.

In [ ]:
issue_json = {
    "id": "ISSUE-1042",
    "projeto": "loja-aurora",
    "autor": "marina.souza",
    "criada_em": "2026-08-14",
    "labels": ["checkout", "pricing", "cliente-impactado"],
    "titulo": "Cupom de desconto e aplicado duas vezes quando o cliente volta para a etapa de pagamento",
    "descricao": (
        "Clientes relataram que o valor final do pedido fica menor do que deveria. "
        "Ao voltar da etapa PAGAMENTO para ENDERECO e avancar de novo, o desconto de 10% "
        "aparece somado duas vezes. O suporte registrou 37 pedidos afetados em 48 horas."
    ),
    "criterios_aceite": [
        "Aplicar o mesmo cupom mais de uma vez nao deve alterar o total alem do primeiro desconto",
        "Voltar e avancar entre ENDERECO e PAGAMENTO deve manter o total estavel",
        "Deve existir teste automatizado que reproduza o defeito e falhe antes da correcao",
        "A correcao deve poder ser desligada sem novo deploy",
    ],
}

issue = Issue.model_validate(issue_json)

print(type(issue).__name__, "|", issue.id, "|", issue.projeto)
print("labels:", issue.labels)
print()
for linha in issue.criterios_numerados():
    print(linha)
print()
print("ids dos criterios:", issue.ids_criterios())

### 2.2 Execucao: uma issue mal formada

Agora uma issue com **quatro** defeitos ao mesmo tempo: falta o `autor`, o titulo e curto
demais, a descricao e curta demais e a lista de criterios esta vazia.

Repare que o Pydantic **reporta todos juntos**. No agente, isso acontece antes de qualquer
chamada ao LLM: dado ruim nao custa dinheiro.

In [ ]:
issue_ruim = {
    "id": "ISSUE-9999",
    "projeto": "loja-aurora",
    "criada_em": "2026-08-14",
    "titulo": "bug",
    "descricao": "nao funciona",
    "criterios_aceite": [],
}

try:
    Issue.model_validate(issue_ruim)
except ValidationError as erro:
    mostrar_erros(erro)

### 2.3 Execucao: o `field_validator` trabalhando

Dois casos que o `min_length=1` sozinho **nao pega**:

1. criterios com espacos e itens em branco -> o validador **limpa** e a issue passa;
2. uma lista que so tem espacos -> tem tamanho 2, passa no `min_length`, mas o validador
   descobre que nao sobrou nada e **recusa**.

In [ ]:
# Caso 1: limpeza
sujo = {**issue_json, "criterios_aceite": ["  total estavel  ", "", "   ", "teste automatizado"]}
print("limpos:", Issue.model_validate(sujo).criterios_aceite)
print()

# Caso 2: so espacos
so_espacos = {**issue_json, "criterios_aceite": ["   ", ""]}
try:
    Issue.model_validate(so_espacos)
except ValidationError as erro:
    mostrar_erros(erro)

---
## 3. Saida do LLM: `Triagem`

Depois que a issue passa pelos guardrails, o LLM classifica a issue. A resposta **nao** e
texto livre: e este contrato.

No projeto, o contrato e entregue ao modelo assim:

```python
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model="claude-opus-5")
triador = llm.with_structured_output(Triagem)   # o schema vira uma "ferramenta"
triagem = triador.invoke(prompt_com_a_issue)    # volta uma instancia de Triagem, ja validada
```

O `with_structured_output` transforma o contrato em JSON Schema, obriga o modelo a responder
preenchendo esse schema e valida a resposta com o Pydantic. Neste lab **nao chamamos o LLM**:
simulamos o JSON que ele devolveria e validamos com o mesmo contrato - o que acontece na
validacao e identico.

**Ferramentas usadas:**

| Recurso | O que garante |
|---|---|
| `Literal[...]` | **fecha o vocabulario**: o modelo nao pode inventar `severidade="urgente"` |
| `Field(description=...)` | nao e comentario: **vai para o schema** e o LLM le como instrucao |
| `Field(default_factory=list)` | `fontes` e opcional |

Vocabulario fechado torna as respostas comparaveis entre execucoes e permite regras
deterministicas depois do modelo (ex.: "severidade alta exige P1").

In [ ]:
class Triagem(BaseModel):
    """Classificacao da issue, produzida pelo LLM."""

    tipo: Literal["bug", "feature", "debito_tecnico", "documentacao", "suporte"]
    severidade: Literal["baixa", "media", "alta", "critica"]
    prioridade: Literal["P0", "P1", "P2", "P3"]
    componentes: List[str] = Field(description="Servicos/modulos afetados, ex.: pricing-service")
    esforco: Literal["XS", "S", "M", "L", "XL"]
    justificativa: str = Field(description="Por que essa classificacao, citando a politica de engenharia")
    fontes: List[str] = Field(
        default_factory=list,
        description="IDs dos trechos da base de conhecimento usados, ex.: politica-de-engenharia#2",
    )

### 3.1 Execucao: o que o LLM realmente "ve"

`model_json_schema()` gera o JSON Schema do contrato - essencialmente o que o
`with_structured_output` envia ao modelo. Repare nos `enum` gerados pelos `Literal`,
nas `description` e na lista de campos obrigatorios.

In [ ]:
schema = Triagem.model_json_schema()

print("severidade:", json.dumps(schema["properties"]["severidade"], ensure_ascii=False))
print("esforco   :", json.dumps(schema["properties"]["esforco"], ensure_ascii=False))
print("fontes    :", json.dumps(schema["properties"]["fontes"], ensure_ascii=False))
print()
print("obrigatorios:", schema["required"])

### 3.2 Execucao: uma resposta valida

O LLM devolve **texto JSON**. `model_validate_json` faz o parse e a validacao num passo so.

In [ ]:
resposta_llm = '''
{
  "tipo": "bug",
  "severidade": "alta",
  "prioridade": "P1",
  "componentes": ["pricing-service", "checkout-web"],
  "esforco": "M",
  "justificativa": "Impacto financeiro direto em 37 pedidos; a politica classifica como severidade alta.",
  "fontes": ["politica-de-engenharia#2", "historico-incidentes#1"]
}
'''

triagem = Triagem.model_validate_json(resposta_llm)

print(f"{issue.id}: {triagem.tipo} | severidade {triagem.severidade} | {triagem.prioridade} | esforco {triagem.esforco}")
print("componentes:", triagem.componentes)
print("fontes     :", triagem.fontes)

### 3.3 Execucao: respostas que o contrato recusa

Tres falhas tipicas de LLM:

1. **vocabulario inventado** - `"urgente"`, `"alta"` como prioridade, `"medio"` como esforco:
   palavras razoaveis para um humano, erros para o contrato;
2. **campo esquecido** - o modelo nao mandou `justificativa` nem `componentes`;
3. **JSON quebrado** - a resposta veio cortada no meio.

Sem o contrato, qualquer uma delas chegaria ao resto do grafo e quebraria la na frente.

In [ ]:
criativa = json.loads(resposta_llm) | {"severidade": "urgente", "prioridade": "alta", "esforco": "medio"}

incompleta = json.loads(resposta_llm)
del incompleta["justificativa"], incompleta["componentes"]

cortada = resposta_llm[:120]

casos = [
    ("vocabulario inventado", lambda: Triagem.model_validate(criativa)),
    ("campo esquecido", lambda: Triagem.model_validate(incompleta)),
    ("JSON quebrado", lambda: Triagem.model_validate_json(cortada)),
]
for nome, validar in casos:
    print(f">>> {nome}")
    try:
        validar()
    except ValidationError as erro:
        mostrar_erros(erro)
    print()

### 3.4 Execucao: regra deterministica sobre a saida

O contrato garante o **formato**; ele nao garante que a resposta faz **sentido**. Como os
valores sao fechados, da para conferir coerencia com um simples dicionario - sem LLM.
A politica de engenharia do DevFlow diz qual prioridade cada severidade exige.

In [ ]:
PRIORIDADE_ESPERADA = {"critica": "P0", "alta": "P1", "media": "P2", "baixa": "P3"}


def conferir_coerencia(t: Triagem) -> list:
    esperada = PRIORIDADE_ESPERADA[t.severidade]   # nunca KeyError: o Literal garante a chave
    if t.prioridade != esperada:
        return [f"severidade {t.severidade} deveria ser {esperada}, veio {t.prioridade}"]
    return []


incoerente = triagem.model_copy(update={"prioridade": "P3"})

print("triagem original :", conferir_coerencia(triagem) or "coerente")
print("triagem incoerente:", conferir_coerencia(incoerente))

---
## 4. Decisao humana: `DecisaoHumana`

No DevFlow o grafo **pausa** antes de terminar (`interrupt()`) e espera uma pessoa.
O que ela responde vira este contrato.

**Ferramenta principal: `Literal`.** Ele **fecha o vocabulario**: so existem tres acoes, e
cada uma corresponde a exatamente um caminho no grafo.

| `acao` | Caminho no grafo |
|---|---|
| `aprovar` | `finalizar` -> fim |
| `revisar` | volta para `planejar` e pausa de novo |
| `rejeitar` | `encerrar` -> fim |

`comentario` e `revisor` tem valor padrao, entao a resposta minima e so `{"acao": "aprovar"}`.

In [ ]:
class DecisaoHumana(BaseModel):
    """O que a pessoa revisora respondeu quando o grafo pausou."""

    acao: Literal["aprovar", "revisar", "rejeitar"]
    comentario: str = ""
    revisor: str = "humano"

### 4.1 Execucao: respostas validas e o erro humano classico

A ultima resposta usa `"aprovado"` em vez de `"aprovar"`. E um erro humano comum - e se
passasse, o roteador do grafo nao saberia para onde ir. Com o contrato, ele e barrado na porta.

In [ ]:
print("minima  :", DecisaoHumana(acao="aprovar").model_dump())
print("completa:", DecisaoHumana(acao="revisar", comentario="detalhar rollback", revisor="ana").model_dump())
print()

for resposta in [{"acao": "aprovado", "revisor": "ana"}, {"comentario": "ok"}]:
    print("resposta:", resposta)
    try:
        DecisaoHumana.model_validate(resposta)
    except ValidationError as erro:
        mostrar_erros(erro)
    print()

### 4.2 Execucao: o contrato escolhendo o caminho

Um roteador simplificado, como o do grafo. Como `acao` so pode ter tres valores, o
dicionario cobre **todos** os casos possiveis - nao existe `else` para um valor inesperado.

In [ ]:
PROXIMO_NO = {"aprovar": "finalizar", "revisar": "planejar", "rejeitar": "encerrar"}


def rotear(resposta: dict) -> str:
    decisao = DecisaoHumana.model_validate(resposta)
    return PROXIMO_NO[decisao.acao]


for resposta in [{"acao": "aprovar"}, {"acao": "revisar", "comentario": "faltou teste"}, {"acao": "rejeitar"}]:
    print(f"{resposta['acao']:<9} -> {rotear(resposta)}")

---
## 5. Ida e volta: `model_dump` e `model_validate_json`

No LangGraph o **estado** do grafo guarda `dict`s, e o checkpointer grava esse estado em
disco para retomar depois da pausa humana - talvez em outro dia, em outro processo.
O ciclo e sempre:

```
objeto --model_dump()--> dict/JSON --(checkpoint)--> dict/JSON --model_validate()--> objeto
```

A execucao abaixo prova que o ciclo nao perde nada - e que o contrato **continua valendo**
quando o dado volta do disco.

In [ ]:
# Serializa como o checkpointer faria
texto = issue.model_dump_json()
print("JSON da issue:", len(texto), "caracteres")
print("igual ao original:", Issue.model_validate_json(texto) == issue)
print()

triagem_json = triagem.model_dump_json()
print("triagem gravada:", triagem_json)
print("igual a original:", Triagem.model_validate_json(triagem_json) == triagem)
print()

# Uma decisao gravada com valor fora do vocabulario (checkpoint adulterado ou codigo antigo)
try:
    DecisaoHumana.model_validate_json('{"acao": "aprovar_sem_revisar", "revisor": "bot"}')
except ValidationError as erro:
    mostrar_erros(erro)

---
## 6. Resumo

| Recurso Pydantic | Onde apareceu | O que resolveu |
|---|---|---|
| campo sem padrao | `Issue`, `DecisaoHumana.acao` | campo obrigatorio |
| `Field(min_length=...)` | `Issue` | entrada minima util |
| `Field(default_factory=list)` | `Issue.labels`, `Triagem.fontes` | listas opcionais sem compartilhar estado |
| `Field(description=...)` | `Triagem` | instrucao que vai **dentro** do schema para o LLM |
| `model_json_schema()` | secao 3.1 | o que o LLM recebe no structured output |
| `@field_validator` | `Issue.criterios_aceite` | regra propria sobre um campo |
| `ValidationError` | secoes 2, 3.3 e 4.1 | todos os erros de uma vez, em vez de quebrar la na frente |
| `Literal[...]` | `Triagem`, `DecisaoHumana.acao` | vocabulario fechado: LLM nao inventa valor, roteamento sem caso inesperado |
| `model_dump` / `model_validate_json` | secao 5 | estado serializavel para checkpoint |

## 7. Exercicios

1. Adicione a `Issue` um `@field_validator` que recuse `id` fora do formato `ISSUE-<numero>`
   (dica: `re.fullmatch(r"ISSUE-\d+", valor)`).
2. `DecisaoHumana` aceita `acao="rejeitar"` sem comentario. Isso faz sentido? Crie um
   `@model_validator(mode="after")` que exija `comentario` quando a acao for `revisar` ou `rejeitar`.
3. Em `Triagem`, adicione um `@model_validator(mode="after")` que recuse `tipo="documentacao"`
   com `severidade="critica"`. Faz sentido essa regra ficar no contrato ou em `conferir_coerencia`?
4. `conferir_coerencia` so **reporta**. Escreva uma versao que devolva uma `Triagem` corrigida
   com a prioridade esperada, e discuta: corrigir a saida do LLM em silencio e seguro?